In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from phentax.waveform import IMRPhenomTHM

FIGSIZE = (10, 6)
import scienceplots
plt.style.use(["science", "notebook"])

## Phentax basis matrix

#### First, create the waveform generator. 

Here we create all the allowed higher modes. The first step is initialize the waveform generator with the desired settings:

In [ ]:
tlowfit = True # use a fit to set the starting time of the root finder used in t(f)
tol = 1e-12 # root finding tolerance

imr = IMRPhenomTHM(
        higher_modes="all",
        include_negative_modes=True, # negative m modes will be produced by simmetry
        t_low_fit=tlowfit,
        coarse_grain=False, # if false it will generate the waveform on a dense time grid with the specified timestep
        atol=tol,
        rtol=tol,
    )

#### generate plus and cross polarizations for one single binary

In [ ]:
m1 = 5e6
m2 = 1e6
chi1 = 0.9
chi2 = 0.3
distance = 500.0
inclination = jnp.pi / 3.0
phi_ref = 0.0
psi = 0.0
f_min = 1e-4
delta_t = 2.5
f_ref = f_min
# t_ref = 0.0

to ensure compatibility with JAX's `vmap` functionality, every output has to have the same shape. 
For this reason, together with times and polarization, we return a mask that for each binary indicates the valid time points.

In [ ]:
times, mask, h_plus, h_cross = imr.compute_polarizations_at_once(
        m1,
        m2,
        chi1,
        chi2,
        distance,
        phi_ref,
        f_ref,
        f_min,
        inclination,
        psi,
        delta_t=delta_t,
    )

In [ ]:
fig = plt.figure(figsize=FIGSIZE)
plt.plot(times[mask], h_plus[mask], label=r"$h_+$")
plt.plot(times[mask], h_cross[mask], label=r"$h_x$")
plt.legend()
plt.xlabel("Time [s]")
plt.ylabel("Strain")
plt.show()

#### we can use the same logic and signature to generate a batch of waveforms. 

For simplicity, here we add a random deviation from the previous parameters.

In [ ]:
num_binaries = 10
num_params = 8
key = jax.random.PRNGKey(0)

key, subkey = jax.random.split(key)
random_params = jax.random.uniform(subkey, (num_binaries, num_params))

m1_batch = ( 1 + 0.7 * random_params[:, 0]) * m1
m2_batch = ( 1 + 0.5 * random_params[:, 1]) * m2
chi1_batch = (1 + 0.1 * random_params[:, 2]) * chi1
chi2_batch = (1 + 0.1 * random_params[:, 3]) * chi2
distance_batch = (1 + 0.1 * random_params[:, 4]) * distance
phi_ref_batch = ( 1 + 0.1 * random_params[:, 5]) * phi_ref
psi_batch = ( 1 + 0.1 * random_params[:, 6]) * psi
inclination_batch = (1 + 0.1 * random_params[:, 7]) * inclination

f_min_batch = jnp.ones(num_binaries) * f_min
f_ref_batch = jnp.ones(num_binaries) * f_ref


In [ ]:
times_batch, mask_batch, h_plus_batch, h_cross_batch = imr.compute_polarizations_at_once(
        m1_batch,
        m2_batch,
        chi1_batch,
        chi2_batch,
        distance_batch,
        phi_ref_batch,
        f_ref,
        f_min,
        inclination_batch,
        psi_batch,
        delta_t=delta_t,
    )

In [ ]:
# plot all the polarizations
fig, axs = plt.subplots(1, 2, figsize=(2 * FIGSIZE[0], FIGSIZE[1]))
for i in range(num_binaries):
    axs[0].plot(times_batch[i][mask_batch[i]], h_plus_batch[i][mask_batch[i]])
    axs[1].plot(times_batch[i][mask_batch[i]], h_cross_batch[i][mask_batch[i]])

axs[0].set_title("Plus polarization")
axs[1].set_title("Cross polarization")
#plt.legend()
axs[0].set_xlabel("Time [s]")
axs[1].set_xlabel("Time [s]")
axs[0].set_ylabel("strain")
plt.show()
    

In [ ]:
# also plot them individually
for i in range(num_binaries):
    fig, axs = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(2 * FIGSIZE[0], FIGSIZE[1]))
    axs[0].plot(times_batch[i][mask_batch[i]], h_plus_batch[i][mask_batch[i]])
    axs[1].plot(times_batch[i][mask_batch[i]], h_cross_batch[i][mask_batch[i]])

    axs[0].set_title("Plus polarization")
    axs[1].set_title("Cross polarization")
    #plt.legend()
    axs[0].set_xlabel("Time [s]")
    axs[1].set_xlabel("Time [s]")
    axs[0].set_ylabel("strain")
    plt.show()

#### Check against PhenomXPY

In [ ]:
from phenomxpy.phenomt.internals import pWF
from phenomxpy.phenomt.phenomt import IMRPhenomTHM as xpy_thm

In [ ]:
batch_idx = 0

m1_here = float(m1_batch[batch_idx])
m2_here = float(m2_batch[batch_idx])
chi1_here = float(chi1_batch[batch_idx])
chi2_here = float(chi2_batch[batch_idx])
distance = float(distance_batch[batch_idx])
inclination_here = float(inclination_batch[batch_idx])
psi_here = float(psi_batch[batch_idx])
phi_ref_here = float(phi_ref_batch[batch_idx])

pwf = pWF(
        eta=m1_here * m2_here / (m1_here + m2_here) ** 2,
        s1=chi1_here,
        s2=chi2_here,
        f_min=f_min,
        f_ref=f_ref,
        total_mass=m1_here + m2_here,
        distance=distance,
        inclination=inclination_here,
        polarization_angle=psi_here,
        delta_t=delta_t,
        phi_ref=phi_ref_here,
    )

mode_array = None

xpy_wave_gen = xpy_thm(mode_array=mode_array, pWF_input=pwf)

xpy_plus, xpy_cross, xpy_times = xpy_wave_gen.compute_polarizations()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(2 * FIGSIZE[0], FIGSIZE[1]))
axs[0].plot(times_batch[0][mask_batch[0]], h_plus_batch[0][mask_batch[0]], label='Phentax')
axs[0].plot(xpy_times, xpy_plus, lw=1.5, ls='--', label='PhenomXPY')
axs[0].legend()

# zoom on the end 
axs[1].plot(times_batch[0][-10000:], h_plus_batch[0][-10000:], label='Phentax')
axs[1].plot(xpy_times[-10000:], xpy_plus[-10000:], lw=1.5, ls='--', label='PhenomXPY')
axs[1].legend()

plt.show()

#### Check the timing as a function of the batch size

In [ ]:
import time
from tqdm import tqdm

In [ ]:
def get_phentax_timings(num_binaries: int, timings: list, key):

    key, subkey = jax.random.split(key)
    random_params = jax.random.uniform(subkey, (num_binaries, num_params))

    m1_batch = ( 1 + 0.7 * random_params[:, 0]) * m1
    m2_batch = ( 1 + 0.5 * random_params[:, 1]) * m2
    chi1_batch = (1 + 0.1 * random_params[:, 2]) * chi1
    chi2_batch = (1 + 0.1 * random_params[:, 3]) * chi2
    distance_batch = (1 + 0.1 * random_params[:, 4]) * distance
    phi_ref_batch = ( 1 + 0.1 * random_params[:, 5]) * phi_ref
    psi_batch = ( 1 + 0.1 * random_params[:, 6]) * psi
    inclination_batch = (1 + 0.1 * random_params[:, 7]) * inclination

    f_min_batch = jnp.ones(num_binaries) * f_min
    f_ref_batch = jnp.ones(num_binaries) * f_ref

    start_time = time.time()

    times_batch, mask_batch, h_plus_batch, h_cross_batch = imr.compute_polarizations_at_once(
        m1_batch,
        m2_batch,
        chi1_batch,
        chi2_batch,
        distance_batch,
        phi_ref_batch,
        f_ref,
        f_min,
        inclination_batch,
        psi_batch,
        delta_t=delta_t,
    )
    h_plus_batch.block_until_ready()

    time_elapsed = time.time() - start_time
    timings.append(time_elapsed)

    return key

In [ ]:
timings = []
num_binaries_all = [1, 5, 10, 15]
for num in tqdm(num_binaries_all):
    key = get_phentax_timings(num, timings, key)

In [ ]:
plt.scatter(num_binaries_all, timings)